# 04 — DistilBERT Full Fine-Tune

Fine-tune DistilBERT on the maintenance work order corpus.

Expected runtime: ~20–30 min on CPU (M1/M2 Mac), ~5 min on GPU.
For faster iteration: use a smaller subset or reduce epochs.

Outputs:
- `models/distilbert_finetuned/` — base model for LoRA comparison in notebook 05
- F1 comparison: TF-IDF vs. DistilBERT

In [ ]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from transformers import (
    DistilBertTokenizerFast, DistilBertForSequenceClassification,
    TrainingArguments, Trainer,
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import sys; sys.path.insert(0, '..')
from src.nlp_pipeline import CATEGORY_LABELS

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
df = pd.read_csv('../data/work_orders.csv')
label2id = {c: i for i, c in enumerate(CATEGORY_LABELS)}
id2label = {i: c for c, i in label2id.items()}
df['label'] = df.failure_category.map(label2id)

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df.label, random_state=42)
train_df, val_df  = train_test_split(train_df, test_size=0.1, stratify=train_df.label, random_state=42)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

In [ ]:
model_name = 'distilbert-base-uncased'
tokenizer  = DistilBertTokenizerFast.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=256, padding='max_length')

train_ds = Dataset.from_pandas(train_df[['text','label']].reset_index(drop=True)).map(tokenize, batched=True)
val_ds   = Dataset.from_pandas(val_df[['text','label']].reset_index(drop=True)).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test_df[['text','label']].reset_index(drop=True)).map(tokenize, batched=True)
train_ds.set_format('torch', columns=['input_ids','attention_mask','label'])

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    model_name, num_labels=len(CATEGORY_LABELS), id2label=id2label, label2id=label2id
).to(device)

# Count trainable parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,} ({n_params/1e6:.1f}M) — 100% of model')

In [ ]:
args = TrainingArguments(
    output_dir='../models/distilbert_finetuned',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=50,
    report_to='none',
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'f1': f1_score(labels, preds, average='macro')}

trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=val_ds,
                  compute_metrics=compute_metrics)

print('Starting fine-tune... (~20 min on CPU, ~5 min on GPU)')
trainer.train()

In [ ]:
# Evaluate on test set
preds_out = trainer.predict(test_ds)
y_pred = np.argmax(preds_out.predictions, axis=-1)
print(classification_report(test_df.label, y_pred, target_names=CATEGORY_LABELS))

# Save model
trainer.save_model('../models/distilbert_finetuned')
tokenizer.save_pretrained('../models/distilbert_finetuned')
print('Saved to models/distilbert_finetuned — proceed to notebook 05 for LoRA comparison.')